In [1]:
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import pandas as pd

os.environ['OPENAI_KEY_SAE'] = '...' # Replace with your OpenAI API key, or with another environment variable (e.g. os.environ['OPENAI_API_KEY'])
os.environ['OPENAI_BASE_URL'] = 'http://0.0.0.0:8010/v1'  # Optional: local OpenAI-compatible endpoint

from hypothesaes.quickstart import train_sae
from hypothesaes.embedding import get_local_embeddings
from hypothesaes.sae import get_multiple_sae_activations
from hypothesaes.select_neurons import select_neurons
from hypothesaes.interpret_neurons import NeuronInterpreter, InterpretConfig, ScoringConfig, SamplingConfig, sample_percentile_bins, ternary_sample_percentile_bins
from hypothesaes.evaluation import score_hypotheses
from hypothesaes.annotate import annotate_texts_with_concepts


In [2]:
current_dir = os.getcwd()
if current_dir.endswith("notebooks"):
    prefix = "../"
else:
    prefix = "./"
base_dir = os.path.join(prefix, "data")

EMBEDDER = "nomic-ai/modernbert-embed-base"
ANNOTATOR_MODEL = "openai/gpt-oss-20b"
INTERPRETER_MODEL = "openai/gpt-oss-120b"
INTERPRET_PROMPT_NAME = "interpret-neuron-binary"

USE_ABS_TOPK = True
USE_VALIDATION = True
DATASET_NAME = "headlines"

EMBED_CACHE_NAME = f"{DATASET_NAME}_{EMBEDDER}"
ACT_CACHE_NAME = EMBED_CACHE_NAME
if USE_ABS_TOPK:
    ACT_CACHE_NAME += "_abstopk"
    INTERPRET_PROMPT_NAME = "interpret-neuron-ternary"
ANN_CACHE_NAME = ACT_CACHE_NAME
if USE_VALIDATION:
    ANN_CACHE_NAME += "_validation"

In [3]:
ACT_CACHE_NAME

'headlines_nomic-ai/modernbert-embed-base_abstopk'

In [4]:


label_col = "label_pairwise"
train_df = pd.read_json(os.path.join(base_dir, DATASET_NAME, "train-8.8K.json"), lines=True)
val_df = pd.read_json(os.path.join(base_dir, DATASET_NAME, "val-1.0K.json"), lines=True)

train_labels = train_df[label_col].values

unique_train_texts = np.unique(train_df["headline_A"].tolist() + train_df["headline_B"].tolist()).tolist()
unique_val_texts = np.unique(val_df["headline_A"].tolist() + val_df["headline_B"].tolist()).tolist()

text2embedding = get_local_embeddings(unique_train_texts + unique_val_texts, model=EMBEDDER, cache_name=EMBED_CACHE_NAME)

train_embeddings = np.stack([text2embedding[text] for text in unique_train_texts])
val_embeddings = np.stack([text2embedding[text] for text in unique_val_texts])

Loading embedding chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Loaded 16468 embeddings in 0.0s


In [5]:
checkpoint_dir = os.path.join(prefix, f"checkpoints/{ACT_CACHE_NAME}")
sae_256_8 = train_sae(embeddings=train_embeddings, M=256, K=8, checkpoint_dir=checkpoint_dir, val_embeddings=val_embeddings, n_epochs=200, patience=5, abs_topk=USE_ABS_TOPK)
sae_32_4 = train_sae(embeddings=train_embeddings, M=32, K=4, checkpoint_dir=checkpoint_dir, val_embeddings=val_embeddings, n_epochs=200, patience=5, abs_topk=USE_ABS_TOPK)

sae_list = [sae_256_8, sae_32_4]

unique_train_activations, neuron_source_sae_info = get_multiple_sae_activations(sae_list, train_embeddings, return_neuron_source_info=True)


Loaded model from ../checkpoints/headlines_nomic-ai/modernbert-embed-base_abstopk/SAE_M=256_K=8.pt onto device cuda
Loaded model from ../checkpoints/headlines_nomic-ai/modernbert-embed-base_abstopk/SAE_M=32_K=4.pt onto device cuda


Computing activations (batchsize=16384):   0%|          | 0/1 [00:00<?, ?it/s]

Computing activations (batchsize=16384):   0%|          | 0/1 [00:00<?, ?it/s]

In [6]:
np.where(unique_train_activations < 0)

(array([    0,     0,     0, ..., 14548, 14548, 14548], shape=(93038,)),
 array([ 67,  78,  91, ..., 232, 257, 287], shape=(93038,)))

In [7]:

headline_A_train = np.stack([text2embedding[text] for text in train_df["headline_A"].tolist()])
headline_B_train = np.stack([text2embedding[text] for text in train_df["headline_B"].tolist()])

Z_A = get_multiple_sae_activations(sae_list, headline_A_train)
Z_B = get_multiple_sae_activations(sae_list, headline_B_train)
Z_delta = Z_A - Z_B

Computing activations (batchsize=16384):   0%|          | 0/1 [00:00<?, ?it/s]

Computing activations (batchsize=16384):   0%|          | 0/1 [00:00<?, ?it/s]

Computing activations (batchsize=16384):   0%|          | 0/1 [00:00<?, ?it/s]

Computing activations (batchsize=16384):   0%|          | 0/1 [00:00<?, ?it/s]

In [8]:
selection_cache_dir = os.path.join(prefix, "selection_cache", ACT_CACHE_NAME)
os.makedirs(selection_cache_dir, exist_ok=True)
selected_neurons, scores = select_neurons(
    activations=Z_delta,
    target=train_labels,
    n_select=20,
    method="lasso",
    classification=True,
    verbose=True,
    cache_dir=selection_cache_dir
)

In [9]:
import importlib
import hypothesaes.interpret_neurons
importlib.reload(hypothesaes.interpret_neurons)
from hypothesaes.interpret_neurons import NeuronInterpreter, ternary_sample_percentile_bins

In [10]:
interpreter = NeuronInterpreter(
    interpreter_model=INTERPRETER_MODEL,
    annotator_model=ANNOTATOR_MODEL,
    n_workers_interpretation=4,
    n_workers_annotation=4,
    cache_name=ACT_CACHE_NAME,
)

n_examples = 20
high_percentile = (80, 100)

if USE_ABS_TOPK:
    interpretation_sampling_config = SamplingConfig(
        function=ternary_sample_percentile_bins,
        n_examples=n_examples,
        sampling_kwargs={
            "high_abs_percentile": high_percentile
        },
    )
else:
    interpretation_sampling_config = SamplingConfig(
        function=sample_percentile_bins,
        n_examples=n_examples,
        sampling_kwargs={
            "high_percentile": high_percentile
        },
    )

interpret_config = InterpretConfig(
    sampling=interpretation_sampling_config,
    n_candidates=5,
    interpretation_prompt_name=INTERPRET_PROMPT_NAME,
    task_specific_instructions="""All of the texts are digital news headlines.
Features should describe specific characteristics in the headline texts. Example features may include:
- "explicitly mentions climate change or another environmental issue"
- "asks the reader a rhetorical question\"""",
)

In [11]:
interpretations = interpreter.interpret_neurons(
    texts=unique_train_texts,
    activations=unique_train_activations,
    neuron_indices=selected_neurons,
    config=interpret_config,
)

[WARNING] < 6 positive samples in bin (80, 100) for neuron 62; using 3 instead
[WARNING] < 6 positive samples in bin (80, 100) for neuron 62; using 3 instead
[WARNING] < 6 positive samples in bin (80, 100) for neuron 62; using 3 instead
[WARNING] < 6 positive samples in bin (80, 100) for neuron 62; using 3 instead
[WARNING] < 6 positive samples in bin (80, 100) for neuron 62; using 3 instead
[WARNING] < 6 negative samples in bin (80, 100) for neuron 243; using 2 instead
[WARNING] < 6 negative samples in bin (80, 100) for neuron 243; using 2 instead
[WARNING] < 6 negative samples in bin (80, 100) for neuron 243; using 2 instead
[WARNING] < 6 negative samples in bin (80, 100) for neuron 243; using 2 instead
[WARNING] < 6 negative samples in bin (80, 100) for neuron 243; using 2 instead


In [12]:
interpretations

{264: ['Contains a strongly sentiment‑laden adjective (positive‑valence adjectives in the positive samples, negative‑valence adjectives in the negative samples, absent from neutral samples)',
  'Uses a conditional clause beginning with the word \\"If\\',
  'Contains an internal period (.) that separates multiple sentences within the headline',
  'Contains a strongly positive evaluative adjective in the positive samples (e.g., “greatest,” “beautiful,” “good,” “YES!”) or a strongly negative evaluative adjective in the negative samples (e.g., “scary,” “false,” “sobbing,” “disappeared”)',
  'Contains a question mark (\u202f?\u202f)'],
 257: ["Contains a question mark ('?')",
  'Mentions the word \\"video\\" (positive) or does not mention the word \\"video\\" (negative)',
  'Mentions a video platform (e.g., the word “Video” or “YouTube”) in positive samples; no mention of any video platform in negative samples',
  'Contains the word “Video” or “Videos” (present in positive samples, absent i

In [13]:
scoring_config = ScoringConfig(
    n_examples=200,
    sampling_function=interpretation_sampling_config.function,
    sampling_kwargs=interpretation_sampling_config.sampling_kwargs
)
all_metrics = interpreter.score_interpretations(
    texts=unique_train_texts,
    activations=unique_train_activations,
    interpretations=interpretations,
    config=scoring_config,
)

[WARNING] < 66 negative samples in bin (80, 100) for neuron 257; using 56 instead
[WARNING] < 66 positive samples in bin (80, 100) for neuron 262; using 30 instead
[WARNING] < 66 negative samples in bin (80, 100) for neuron 140; using 13 instead
[WARNING] < 66 positive samples in bin (80, 100) for neuron 67; using 19 instead
[WARNING] < 66 negative samples in bin (80, 100) for neuron 41; using 21 instead
[WARNING] < 66 negative samples in bin (80, 100) for neuron 46; using 15 instead
[WARNING] < 66 positive samples in bin (80, 100) for neuron 267; using 46 instead
[WARNING] < 66 positive samples in bin (80, 100) for neuron 276; using 32 instead
[WARNING] < 66 negative samples in bin (80, 100) for neuron 139; using 21 instead
[WARNING] < 66 negative samples in bin (80, 100) for neuron 260; using 50 instead
[WARNING] < 66 positive samples in bin (80, 100) for neuron 62; using 3 instead
[WARNING] < 66 negative samples in bin (80, 100) for neuron 210; using 7 instead
[WARNING] < 66 negativ

In [14]:
results = []
for neuron_idx, fidelity_scores in all_metrics.items():
    best_interp = max(interpretations[neuron_idx], key=lambda interp: fidelity_scores[interp]['f1'])
    results.append({
        'neuron_idx': neuron_idx,
        'source_sae': neuron_source_sae_info[neuron_idx],
        'target_lasso_coef': scores[selected_neurons.index(neuron_idx)],
        'interpretation': best_interp,
        'f1_fidelity_score': fidelity_scores[best_interp]['f1'],
    })

hypothesis_df = pd.DataFrame(results)
pd.set_option('display.max_colwidth', None)
display(hypothesis_df.sort_values(by='target_lasso_coef', ascending=False))
pd.reset_option('display.max_colwidth')

,neuron_idx,source_sae,target_lasso_coef,interpretation,f1_fidelity_score
1,257,"(SAE_1, 32, 4)",0.158348,"Contains the word “Video” or “Videos” (present in positive samples, absent in negative and neutral samples)",0.862069
2,262,"(SAE_1, 32, 4)",0.109705,"Mentions a second‑person pronoun (you/your) (positive) or mentions an environmental/climate‑related term (e.g., climate, carbon, fossil fuel, global warming, clean water) (negative)",0.519481
3,140,"(SAE_0, 256, 8)",0.069642,"Mentions visual media (photos/graphics/images) in positive samples and mentions non‑visual tools or technologies (apps, ingredients, tattoos) in negative samples",0.916031
5,41,"(SAE_0, 256, 8)",0.050278,Conveys a negative sentiment in the headline (positive samples) versus a positive sentiment (negative samples),0.916031
6,46,"(SAE_0, 256, 8)",0.039174,"Mentions a specific quantity or numeral (e.g., '1', 'One' in positive samples; '7' etc. in negative samples)",0.601942
9,274,"(SAE_1, 32, 4)",0.020018,"References spoken language (e.g., words, speech, say, hearing) in positive samples versus references visual media/film (e.g., Hollywood, TV, movies, film) in negative samples",0.844444
10,139,"(SAE_0, 256, 8)",0.017634,Headline uses title case (each word capitalized),0.822695
11,260,"(SAE_1, 32, 4)",0.016116,Contains a digit (0‑9) in the first clause of the headline (before any period),0.842105
12,62,"(SAE_0, 256, 8)",0.015230,Contains a negation word such as “not” or a contraction ending in “n't”,0.187500
13,210,"(SAE_0, 256, 8)",0.015022,Contains the word “Response”,0.810811


## Training Predictor

In [15]:
def display_df(m, evaluation_df):
    print(f"AUC: {m['auroc']:.3f}")
    print(f"Significant hypotheses: {m['Significant'][0]}/{m['Significant'][1]} " 
          f"(p < {m['Significant'][2]:.3e})")
    pd.set_option('display.max_colwidth', None)
    display(evaluation_df.sort_values(by='separation_score', ascending=False).round(4))
    pd.reset_option('display.max_colwidth')
    

In [16]:
Z_preproc = np.where(np.abs(Z_delta) > 1e-6, Z_delta, 0)
Z_preproc = np.sign(Z_preproc)
train_delta_annots = {}
for _, row in hypothesis_df.iterrows():
    hyp, neuron_idx = row[['interpretation', 'neuron_idx']]
    hyp_labels = Z_preproc[:, neuron_idx]
    train_delta_annots[hyp] = hyp_labels

In [17]:
_, _, train_predictor = score_hypotheses(
    hypothesis_annotations=train_delta_annots,
    y_true=train_labels,
    classification=True,
)

Optimization terminated successfully.
         Current function value: 0.641275
         Iterations 5


## Evaluating the Predictor

In [18]:
if USE_VALIDATION:
    holdout_basename = "val-1.0K.json"
else:
    holdout_basename = "holdout-4.4K.json"
holdout_df = pd.read_json(os.path.join(base_dir, DATASET_NAME, holdout_basename), lines=True)

headline_A_holdout = holdout_df["headline_A"].tolist()
headline_B_holdout = holdout_df["headline_B"].tolist()
labels_holdout = holdout_df[label_col].values

hypotheses = hypothesis_df["interpretation"].tolist()

In [19]:
ANN_CACHE_NAME

'headlines_nomic-ai/modernbert-embed-base_abstopk_validation'

In [ ]:
headline_A_annotations = annotate_texts_with_concepts(
    texts=headline_A_holdout,
    concepts=hypotheses,
    cache_name=ANN_CACHE_NAME,
    model=ANNOTATOR_MODEL,
    n_workers=100
)

Found 2080 cached items; annotating 18300 uncached items


Annotating:   0%|          | 0/18300 [00:00<?, ?it/s]

In [ ]:

headline_B_annotations = annotate_texts_with_concepts(
    texts=headline_B_holdout,
    concepts=hypotheses,
    cache_name=ANN_CACHE_NAME,
    model=ANNOTATOR_MODEL,
    n_workers=100
)


In [ ]:
eval_delta_annots = {key : headline_A_annotations[key] - headline_B_annotations[key] for key in headline_A_annotations.keys()}
metrics, evaluation_df, _ = score_hypotheses(
    hypothesis_annotations=eval_delta_annots,
    y_true=labels_holdout,
    classification=True,
    predictor=train_predictor
)

In [ ]:
display_df(metrics, evaluation_df)